# Trail-4 Test Set Evaluation

**Objective:** Evaluate the trained Trail-4 model on the test set

**Training Results:**
- Best PSNR: 21.28 dB
- Training time: 3.52 hours (100 epochs)
- Improvement over Trail-3: +8.68 dB

**Model Checkpoint:**
```
/content/drive/MyDrive/October Thesis Dataset/trail4_checkpoints/trail4_best.pt
```

**Metrics to Calculate:**
1. PSNR (Peak Signal-to-Noise Ratio)
2. Correlation (Pearson)
3. SSIM (Structural Similarity Index)
4. R-peak amplitude error
5. QRS width error
6. SNR (Signal-to-Noise Ratio)
7. MAE (Mean Absolute Error)
8. RMSE (Root Mean Square Error)

## Step 1: Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install dependencies
!pip install -q scipy scikit-learn scikit-image tqdm
print("✅ Setup complete")

## Step 2: Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import glob
import os
from scipy.stats import pearsonr
import scipy.signal as signal
from skimage.metrics import structural_similarity as compare_ssim
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

## Step 3: Configuration

In [ ]:
CONFIG = {
    'checkpoint_path': '/content/drive/MyDrive/October Thesis Dataset/trail4_checkpoints/trail4_best.pt',
    'chunk_dir': '/content/drive/MyDrive/October Thesis Dataset/Refine-Chunked',
    'split_file': '/content/drive/MyDrive/October Thesis Dataset/Refine-Chunked/train_val_test_split.json',
    'results_dir': '/content/drive/MyDrive/October Thesis Dataset/trail4_results',
    'sample_plots_dir': '/content/drive/MyDrive/October Thesis Dataset/trail4_results/sample_plots',
    'fix_polarity': True,  # Same as training
}

os.makedirs(CONFIG['results_dir'], exist_ok=True)
os.makedirs(CONFIG['sample_plots_dir'], exist_ok=True)

print("Configuration:")
print(f"  Checkpoint: {CONFIG['checkpoint_path']}")
print(f"  Results: {CONFIG['results_dir']}")
print(f"  Polarity fix: {CONFIG['fix_polarity']}")

## Step 4: Polarity Correction Function

In [ ]:
def ensure_positive_polarity(ecg):
    """Ensure ECG R-peaks point upward (positive direction)"""
    max_val = np.max(ecg)
    min_val = np.min(ecg)
    if np.abs(min_val) > np.abs(max_val):
        ecg = -ecg
    return ecg

print("✅ Polarity correction function ready")

## Step 5: Metric Functions

In [ ]:
def calc_psnr(real, fake):
    """Calculate PSNR"""
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    max_val = np.max(np.abs(real))
    return 20 * np.log10(max_val / np.sqrt(mse))

def calc_snr(real, fake):
    """Calculate SNR"""
    signal_power = np.mean(real ** 2)
    noise_power = np.mean((real - fake) ** 2)
    if noise_power < 1e-10:
        return 100.0
    return 10 * np.log10(signal_power / noise_power)

def calc_correlation(real, fake):
    """Calculate Pearson correlation"""
    corr, _ = pearsonr(real.flatten(), fake.flatten())
    return corr

def calc_ssim(real, fake):
    """Calculate SSIM"""
    real_norm = (real - real.min()) / (real.max() - real.min() + 1e-8)
    fake_norm = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return compare_ssim(real_norm, fake_norm, data_range=1.0)

def detect_r_peaks(ecg_signal, fs=8000, distance=400, prominence=0.5):
    """Detect R-peaks"""
    peaks, properties = signal.find_peaks(
        ecg_signal,
        distance=distance,
        prominence=prominence
    )
    return peaks, properties

def calc_r_peak_error(real, fake):
    """Calculate R-peak amplitude error"""
    try:
        real_peaks, _ = detect_r_peaks(real)
        if len(real_peaks) == 0:
            return np.nan
        real_amps = real[real_peaks]
        fake_amps = fake[real_peaks]
        return np.mean(np.abs(real_amps - fake_amps))
    except:
        return np.nan

def calc_qrs_width(ecg_signal, r_peak_idx, fs=8000, threshold=0.5):
    """Calculate QRS width for a single R-peak"""
    r_amp = ecg_signal[r_peak_idx]
    threshold_amp = threshold * abs(r_amp)
    
    q_idx = r_peak_idx
    search_range = min(200, r_peak_idx)
    for i in range(r_peak_idx - 1, r_peak_idx - search_range, -1):
        if abs(ecg_signal[i]) < threshold_amp:
            q_idx = i
            break
    
    s_idx = r_peak_idx
    search_range = min(200, len(ecg_signal) - r_peak_idx - 1)
    for i in range(r_peak_idx + 1, r_peak_idx + search_range):
        if abs(ecg_signal[i]) < threshold_amp:
            s_idx = i
            break
    
    width_ms = ((s_idx - q_idx) / fs) * 1000
    return width_ms

def calc_qrs_width_error(real, fake):
    """Calculate QRS width error"""
    try:
        real_peaks, _ = detect_r_peaks(real)
        if len(real_peaks) == 0:
            return np.nan
        
        errors = []
        for r_peak in real_peaks:
            real_width = calc_qrs_width(real, r_peak)
            fake_width = calc_qrs_width(fake, r_peak)
            errors.append(abs(real_width - fake_width))
        return np.mean(errors)
    except:
        return np.nan

def calc_mae(real, fake):
    """Calculate Mean Absolute Error"""
    return np.mean(np.abs(real - fake))

def calc_rmse(real, fake):
    """Calculate Root Mean Square Error"""
    return np.sqrt(np.mean((real - fake) ** 2))

print("✅ All metric functions ready")

## Step 6: Load Architecture (Same as Training)

In [ ]:
# Copy the entire architecture from TRAIL4_COMPLETE.ipynb
# (MultiHeadSelfAttention, MultiScaleDilatedBlock, EnhancedResidualBlock, ImprovedGeneratorNoTanh)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = torch.nn.functional.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        out = attn @ v
        out = out.transpose(2, 3).reshape(B, C, T)
        out = self.proj(out)
        return out + x


class MultiScaleDilatedBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (channels % 3 > 0)
        c2 = channels // 3 + (channels % 3 > 1)
        c3 = channels // 3
        
        self.conv_d1 = nn.Conv1d(channels, c1, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, kernel_size=3, padding=4, dilation=4)
        
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        d1 = self.conv_d1(x)
        d2 = self.conv_d2(x)
        d4 = self.conv_d4(x)
        concat = torch.cat([d1, d2, d4], dim=1)
        out = self.fusion(concat)
        return out + x


class EnhancedResidualBlock(nn.Module):
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


class ImprovedGeneratorNoTanh(nn.Module):
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        
        self.scenario_embed = nn.Linear(num_scenarios, base_channels)
        
        self.enc1 = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True)
        )
        self.enc3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True)
        )
        self.enc4 = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )
        self.enc5 = nn.Sequential(
            nn.Conv1d(base_channels * 8, base_channels * 8, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )
        
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(base_channels * 8, use_attention=True, use_dilated=True)
        )
        
        self.dec5 = nn.Sequential(
            nn.ConvTranspose1d(base_channels * 8, base_channels * 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 8),
            nn.ReLU(inplace=True)
        )
        self.dec4 = nn.Sequential(
            nn.Conv1d(base_channels * 16, base_channels * 8, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 4),
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Sequential(
            nn.Conv1d(base_channels * 8, base_channels * 4, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels * 2),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 2, kernel_size=1),
            nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True)
        )
        self.dec1 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels, kernel_size=1),
            nn.ConvTranspose1d(base_channels, 1, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(1)
        )
    
    def forward(self, x, scenario_onehot):
        scenario_emb = self.scenario_embed(scenario_onehot).unsqueeze(2)
        
        e1 = self.enc1(x) + scenario_emb
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        
        b = self.bottleneck(e5)
        
        d5 = self.dec5(b)
        d5 = torch.cat([d5, e4], dim=1)
        
        d4 = self.dec4(d5)
        d4 = torch.cat([d4, e3], dim=1)
        
        d3 = self.dec3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        
        d2 = self.dec2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        
        return self.dec1(d2)

print("✅ Architecture loaded")

## Step 7: Load Model Checkpoint

In [ ]:
print("Loading model...")
generator = ImprovedGeneratorNoTanh().to(device)

checkpoint = torch.load(CONFIG['checkpoint_path'], map_location=device, weights_only=False)
generator.load_state_dict(checkpoint['generator_state_dict'])
generator.eval()

print(f"✅ Model loaded from checkpoint")
print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"   Training PSNR: {checkpoint.get('psnr', 'N/A'):.2f} dB")
print(f"   Training Correlation: {checkpoint.get('correlation', 'N/A'):.3f}")
print(f"   Training SSIM: {checkpoint.get('ssim', 'N/A'):.3f}")

## Step 8: Load Test Data

In [ ]:
print("Loading test data...")

with open(CONFIG['split_file'], 'r') as f:
    splits = json.load(f)
test_subjects = set(splits['test_subjects'])

all_chunks = sorted(glob.glob(os.path.join(CONFIG['chunk_dir'], '*.npz')))
test_chunks = []
test_scenarios = []
test_subjects_list = []

for path in all_chunks:
    try:
        data = np.load(path, allow_pickle=True)
        if str(data['subject_id']) in test_subjects:
            test_chunks.append(path)
            test_subjects_list.append(str(data['subject_id']))
            
            scenario_str = str(data['scenario']).lower()
            if 'rest' in scenario_str or 'laying' in scenario_str:
                scenario = 0
            elif 'exercise' in scenario_str or 'bicycle' in scenario_str:
                scenario = 1
            else:
                scenario = 2
            test_scenarios.append(scenario)
    except:
        pass

print(f"✅ Test set: {len(test_chunks)} chunks from {len(test_subjects)} subjects")
print(f"   Scenario distribution:")
print(f"     Rest: {test_scenarios.count(0)} chunks")
print(f"     Exercise: {test_scenarios.count(1)} chunks")
print(f"     Other: {test_scenarios.count(2)} chunks")

## Step 9: Generate Predictions on Test Set

In [ ]:
print("Generating predictions on test set...")
print("This may take 5-10 minutes...")

results = []

with torch.no_grad():
    for idx in tqdm(range(len(test_chunks)), desc="Evaluating"):
        try:
            # Load data
            data = np.load(test_chunks[idx], allow_pickle=True)
            ecg_real = data['ecg'].astype(np.float32)
            pcg = data['pcg'].astype(np.float32)
            scenario = test_scenarios[idx]
            subject_id = test_subjects_list[idx]
            
            # Apply polarity fix
            if CONFIG['fix_polarity']:
                ecg_real = ensure_positive_polarity(ecg_real)
            
            # Store original (unnormalized) for metrics
            ecg_real_original = ecg_real.copy()
            
            # Normalize
            ecg_real = (ecg_real - ecg_real.mean()) / (ecg_real.std() + 1e-8)
            pcg = (pcg - pcg.mean()) / (pcg.std() + 1e-8)
            
            # Prepare input
            pcg_tensor = torch.from_numpy(pcg).unsqueeze(0).unsqueeze(0).to(device)
            scenario_onehot = np.zeros(3, dtype=np.float32)
            scenario_onehot[scenario] = 1.0
            scenario_tensor = torch.from_numpy(scenario_onehot).unsqueeze(0).to(device)
            
            # Generate
            ecg_fake = generator(pcg_tensor, scenario_tensor)
            ecg_fake_np = ecg_fake.squeeze().cpu().numpy()
            
            # Calculate metrics on normalized signals
            psnr = calc_psnr(ecg_real, ecg_fake_np)
            snr = calc_snr(ecg_real, ecg_fake_np)
            correlation = calc_correlation(ecg_real, ecg_fake_np)
            ssim = calc_ssim(ecg_real, ecg_fake_np)
            mae = calc_mae(ecg_real, ecg_fake_np)
            rmse = calc_rmse(ecg_real, ecg_fake_np)
            
            # Denormalize generated ECG for physiological metrics
            ecg_fake_denorm = ecg_fake_np * (ecg_real_original.std() + 1e-8) + ecg_real_original.mean()
            
            # Calculate physiological metrics on denormalized signals
            r_peak_error = calc_r_peak_error(ecg_real_original, ecg_fake_denorm)
            qrs_width_error = calc_qrs_width_error(ecg_real_original, ecg_fake_denorm)
            
            results.append({
                'chunk': os.path.basename(test_chunks[idx]),
                'subject_id': subject_id,
                'scenario': scenario,
                'psnr': psnr,
                'snr': snr,
                'correlation': correlation,
                'ssim': ssim,
                'mae': mae,
                'rmse': rmse,
                'r_peak_error': r_peak_error,
                'qrs_width_error': qrs_width_error,
            })
        except Exception as e:
            print(f"Error processing {test_chunks[idx]}: {e}")
            continue

print(f"\n✅ Evaluation complete: {len(results)}/{len(test_chunks)} chunks processed")

## Step 10: Calculate Summary Statistics

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

# Remove NaN values for physiological metrics
df_clean = df.dropna(subset=['psnr', 'correlation', 'ssim'])

print("="*70)
print("TRAIL-4 TEST SET RESULTS")
print("="*70)
print(f"\nSamples: {len(df_clean)} / {len(df)} (after removing invalid)")
print(f"\n{'Metric':<20} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("-"*70)

metrics_to_show = ['psnr', 'snr', 'correlation', 'ssim', 'mae', 'rmse', 'r_peak_error', 'qrs_width_error']
metric_names = ['PSNR (dB)', 'SNR (dB)', 'Correlation', 'SSIM', 'MAE', 'RMSE', 'R-peak Error (mV)', 'QRS Width Error (ms)']

summary_stats = {}
for metric, name in zip(metrics_to_show, metric_names):
    values = df_clean[metric].dropna()
    if len(values) > 0:
        mean_val = values.mean()
        std_val = values.std()
        min_val = values.min()
        max_val = values.max()
        print(f"{name:<20} {mean_val:<12.3f} {std_val:<12.3f} {min_val:<12.3f} {max_val:<12.3f}")
        summary_stats[metric] = {
            'mean': float(mean_val),
            'std': float(std_val),
            'min': float(min_val),
            'max': float(max_val)
        }

print("="*70)

# Results by scenario
print("\nResults by Scenario:")
print("-"*70)
scenario_names = ['Rest', 'Exercise', 'Other']
for scenario_id, scenario_name in enumerate(scenario_names):
    df_scenario = df_clean[df_clean['scenario'] == scenario_id]
    if len(df_scenario) > 0:
        print(f"\n{scenario_name} (n={len(df_scenario)}):")
        print(f"  PSNR: {df_scenario['psnr'].mean():.2f} ± {df_scenario['psnr'].std():.2f} dB")
        print(f"  Correlation: {df_scenario['correlation'].mean():.3f} ± {df_scenario['correlation'].std():.3f}")
        print(f"  SSIM: {df_scenario['ssim'].mean():.3f} ± {df_scenario['ssim'].std():.3f}")

print("="*70)

## Step 11: Save Results

In [ ]:
# Save detailed results
df.to_csv(os.path.join(CONFIG['results_dir'], 'trail4_test_detailed_results.csv'), index=False)
print(f"✅ Detailed results saved to: trail4_test_detailed_results.csv")

# Save summary statistics
summary = {
    'test_samples': len(df_clean),
    'total_samples': len(df),
    'metrics': summary_stats,
    'by_scenario': {}
}

for scenario_id, scenario_name in enumerate(['rest', 'exercise', 'other']):
    df_scenario = df_clean[df_clean['scenario'] == scenario_id]
    if len(df_scenario) > 0:
        summary['by_scenario'][scenario_name] = {
            'n_samples': len(df_scenario),
            'psnr_mean': float(df_scenario['psnr'].mean()),
            'psnr_std': float(df_scenario['psnr'].std()),
            'correlation_mean': float(df_scenario['correlation'].mean()),
            'correlation_std': float(df_scenario['correlation'].std()),
            'ssim_mean': float(df_scenario['ssim'].mean()),
            'ssim_std': float(df_scenario['ssim'].std())
        }

with open(os.path.join(CONFIG['results_dir'], 'trail4_test_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary statistics saved to: trail4_test_summary.json")

print("\n" + "="*70)
print("TEST EVALUATION COMPLETE")
print("="*70)
print(f"\nFiles saved to: {CONFIG['results_dir']}")
print("  - trail4_test_detailed_results.csv")
print("  - trail4_test_summary.json")

## Step 12: Comparison with Training Results

In [ ]:
print("="*70)
print("TRAINING vs TEST COMPARISON")
print("="*70)
print(f"\n{'Metric':<20} {'Training':<15} {'Test':<15} {'Difference':<15}")
print("-"*70)

# Training metrics from checkpoint
train_psnr = checkpoint.get('psnr', 21.28)
train_corr = checkpoint.get('correlation', 0.184)
train_ssim = checkpoint.get('ssim', 0.865)

# Test metrics
test_psnr = summary_stats['psnr']['mean']
test_corr = summary_stats['correlation']['mean']
test_ssim = summary_stats['ssim']['mean']

print(f"{'PSNR (dB)':<20} {train_psnr:<15.2f} {test_psnr:<15.2f} {test_psnr-train_psnr:<+15.2f}")
print(f"{'Correlation':<20} {train_corr:<15.3f} {test_corr:<15.3f} {test_corr-train_corr:<+15.3f}")
print(f"{'SSIM':<20} {train_ssim:<15.3f} {test_ssim:<15.3f} {test_ssim-train_ssim:<+15.3f}")
print("="*70)

if abs(test_psnr - train_psnr) < 2.0:
    print("\n✅ Model generalizes well! Test PSNR within 2 dB of training.")
else:
    print("\n⚠️ Significant difference between training and test PSNR.")
    print("   This may indicate overfitting or train/test distribution mismatch.")

## Done!

### Trail-4 Test Evaluation Complete

**Results saved to:**
- `trail4_test_detailed_results.csv` - Per-sample metrics
- `trail4_test_summary.json` - Summary statistics

**Next steps:**
1. Run visualization notebook to generate sample plots
2. Run comparison notebook to compare Trail-2, Trail-3, and Trail-4
3. Include results in thesis and presentation